  ENGINEERING ANALYSIS
  

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv("ev6_feature_engineering.csv")

In [3]:
df.describe().head(20)

,Vehicle_Speed_kmh,Speed_mps,Accel_X_mps2,Accel_Y_mps2,Accel_Z_mps2,Altitude_m,Road_Grade_pct,Heading_deg,Pitch_deg,Roll_deg,...,Battery_Power_Calculated_kW,TPMS_Avg_Calculated,Cal_cell_voltage_spread,Cell_Spread_Difference,Battery_Temp_Spread_Calculated_C,Battery_Temp_Spread_Difference,Speed_Calculated_mps,Speed_Difference_mps,Road_Load_Calculated_N,Road_Load_Difference_N
count,11985.000000,11953.000000,12008.000000,12008.000000,12008.000000,11985.000000,12035.000000,11985.000000,11985.000000,11985.000000,...,11869.000000,11804.000000,11870.000000,1.170200e+04,11702.000000,11702.0,11985.000000,11953.000000,12080.000000,1.208000e+04
mean,13.976234,3.891601,0.008935,0.548574,10.029965,97.394637,0.281000,163.784006,-0.034528,0.344331,...,9.483512,34.763057,0.007219,-6.222294e-18,1.730217,0.0,3.882287,0.000625,306.367761,1.821192e-07
std,10.662591,2.960378,0.791951,0.448557,0.078432,68.160097,6.668213,92.239547,1.284656,1.390392,...,17.031081,5.591918,0.009619,1.648804e-16,0.999158,0.0,2.961831,0.056236,2122.310738,6.079175e-05
min,-5.000000,0.000000,-3.525500,-3.025500,8.875000,1.500000,-15.000000,0.000000,-11.320000,-7.680000,...,-86.740000,0.000000,0.000000,-4.614364e-16,0.000000,0.0,-1.388889,-0.853344,-8460.022600,-2.000000e-04
25%,5.317000,1.493300,-0.484700,0.334200,9.992500,59.760000,-3.433500,100.060000,-0.640000,-0.440000,...,0.620000,33.950000,0.000000,-1.734723e-17,1.000000,0.0,1.476944,-0.000022,-1029.278850,-1.136868e-13
50%,11.562800,3.222600,-0.125000,0.553600,10.033200,87.700000,0.033000,173.440000,0.000000,0.460000,...,6.880000,35.400000,0.000000,0.000000e+00,2.000000,0.0,3.211889,0.000000,239.505700,0.000000e+00
75%,21.185400,5.889500,0.538300,0.752500,10.070000,98.460000,3.900450,218.875000,0.600000,1.260000,...,18.380000,37.350000,0.020000,0.000000e+00,2.000000,0.0,5.884833,0.000022,1796.819175,2.273737e-13
max,38.317300,10.643700,3.449000,3.617300,10.676000,396.575000,15.000000,359.900000,8.160000,6.150000,...,175.180000,40.350000,0.040000,4.267420e-16,6.000000,0.0,10.643694,4.357389,8303.276100,2.000000e-04


In [ ]:
dynamics_summary = df.groupby("Driving_Behaviour").agg(
    Avg_Speed_kmh=("Vehicle_Speed_kmh", "mean"),
    Max_Speed_kmh=("Vehicle_Speed_kmh", "max"),
    Avg_Accel_mps2=("Accel_X_mps2", "mean"),
    Avg_Road_Grade_pct=("Road_Grade_pct", "mean")
).reset_index()

print(dynamics_summary.round(2))

  Driving_Behaviour  Avg_Speed_kmh  Max_Speed_kmh  Avg_Accel_mps2  \
0      Accelerating          14.85          38.32            1.08   
1           Braking          10.89          36.95           -0.94   
2          Coasting          15.00          37.95           -0.09   
3           Unknown           0.70           3.40             NaN   

   Avg_Road_Grade_pct  
0                0.80  
1               -1.22  
2                0.71  
3                4.04  


ROAD LOAD WITH DRIVING CONDITION

In [11]:
road_load_summary = df.groupby(
    ["Driving_Behaviour", "Dominant_Road_Load_Component"]
).size().reset_index(name="Record_Count")

print(road_load_summary)

  Driving_Behaviour Dominant_Road_Load_Component  Record_Count
0      Accelerating                Grade_Force_N           346
1      Accelerating             Inertial_Force_N          2765
2      Accelerating         Rolling_Resistance_N             2
3           Braking                Grade_Force_N           707
4           Braking             Inertial_Force_N          2156
5          Coasting                Grade_Force_N          3733
6          Coasting             Inertial_Force_N          1967
7          Coasting         Rolling_Resistance_N           332
8           Unknown                Grade_Force_N            28
9           Unknown         Rolling_Resistance_N            44


Battery Power vs Driving Behaviour

In [12]:
ev_power_summary = df.groupby("Driving_Behaviour").agg(
    Avg_Battery_Power_kW=("Battery_Power_Calculated_kW", "mean"),
    Max_Battery_Power_kW=("Battery_Power_Calculated_kW", "max"),
    Min_Battery_Power_kW=("Battery_Power_Calculated_kW", "min"),
    Avg_SOC_pct=("Battery_SOC_pct", "mean")
).reset_index()

display(ev_power_summary.round(2))

,Driving_Behaviour,Avg_Battery_Power_kW,Max_Battery_Power_kW,Min_Battery_Power_kW,Avg_SOC_pct
0,Accelerating,16.20,174.63,-80.10,71.96
1,Braking,-1.84,175.18,-61.70,69.37
2,Coasting,11.46,109.35,-86.74,73.97
3,Unknown,2.47,17.13,0.42,77.59


SOC change per session

In [13]:
soc_summary = df.groupby("Session_ID").agg(
    Start_SOC_pct=("Battery_SOC_pct", "first"),
    End_SOC_pct=("Battery_SOC_pct", "last")
).reset_index()

soc_summary["SOC_Change_pct"] = (
    soc_summary["End_SOC_pct"] -
    soc_summary["Start_SOC_pct"]
)

display(soc_summary.round(2))

,Session_ID,Start_SOC_pct,End_SOC_pct,SOC_Change_pct
0,EV6_628BDF72,93.5,93.5,0.0
1,EV6_628BDF90,93.5,93.5,0.0
2,EV6_628BE0B8,93.5,92.5,-1.0
3,EV6_628BE10F,92.5,92.5,0.0
4,EV6_628CEDDB,92.5,92.0,-0.5
...,...,...,...,...
145,EV6_64CBC5D8,89.5,89.0,-0.5
146,EV6_64CBC5ED,89.0,87.0,-2.0
147,EV6_64CBC5FE,87.0,86.5,-0.5
148,EV6_64CBC601,86.5,85.0,-1.5


Battery Temperature Analysis

In [14]:
temp_summary = df.groupby("Driving_Behaviour").agg(
    Avg_Battery_Temp_C=("Battery_Temp_C", "mean"),
    Max_Battery_Temp_C=("Battery_Temp_C", "max"),
    Min_Battery_Temp_C=("Battery_Temp_C", "min"),
    Avg_Temp_Spread_C=("Battery_Temp_Spread_C", "mean")
).reset_index()

display(temp_summary.round(2))

,Driving_Behaviour,Avg_Battery_Temp_C,Max_Battery_Temp_C,Min_Battery_Temp_C,Avg_Temp_Spread_C
0,Accelerating,21.05,31.0,0.0,1.72
1,Braking,9.91,29.0,0.0,1.46
2,Coasting,14.35,31.0,-1.0,1.86
3,Unknown,11.57,27.0,0.0,1.51


In [15]:
df.to_csv("ev6_mastercleaned.csv",index=False)